# Stage 1: Basic Data Structure with Smart Display

In [8]:
class Given:

    def __init__(self, key):  # constructor
        self.key = key
        self.right = None
        self.left = None

    def display(self):  # this is the display function (uses a helper, below)
        lines, *_ = self._display_aux()

        for line in lines:
            print(line)

    def _display_aux(self):  # the recursive helper
        """Returns list of strings, width, height, and horizontal coordinate of the root."""

        # No child.
        if self.right is None and self.left is None:
            line = '%s' % self.key
            width = len(line)
            height = 1
            middle = width // 2

            return [line], width, height, middle

        # Only left child.
        if self.right is None:
            lines, n, p, x = self.left._display_aux()
            s = '%s' % self.key
            u = len(s)

            first_line = (x + 1) * ' ' + (n - x - 1) * '_' + s
            second_line = x * ' ' + '/' + (n - x - 1 + u) * ' '

            shifted_lines = [line + u * ' ' for line in lines]

            return [first_line, second_line] + shifted_lines, n + u, p + 2, n + u // 2

        # Only right child.
        if self.left is None:
            lines, n, p, x = self.right._display_aux()
            s = '%s' % self.key
            u = len(s)

            first_line = s + x * '_' + (n - x) * ' '
            second_line = (u + x) * ' ' + '\\' + (n - x - 1) * ' '

            shifted_lines = [u * ' ' + line for line in lines]

            return [first_line, second_line] + shifted_lines, n + u, p + 2, u // 2

        # Two children.
        left, n, p, x = self.left._display_aux()
        right, m, q, y = self.right._display_aux()

        s = '%s' % self.key
        u = len(s)

        first_line = (
            (x + 1) * ' '
            + (n - x - 1) * '_'
            + s
            + y * '_'
            + (m - y) * ' '
        )

        second_line = (
            x * ' '
            + '/'
            + (n - x - 1 + u + y) * ' '
            + '\\'
            + (m - y - 1) * ' '
        )

        if p < q:
            left += [n * ' '] * (q - p)

        elif q < p:
            right += [m * ' '] * (p - q)

        zipped_lines = zip(left, right)

        lines = [
            first_line,
            second_line
        ] + [
            a + u * ' ' + b
            for a, b in zipped_lines
        ]

        return lines, n + m + u, max(p, q) + 2, n + u // 2


if __name__ == "__main__":  # do not run this during import from another file

    b = Given(1)

    print("-------------", 1)
    b.display()

    print("end of first test")

    # first test
    c = Given(3)

    print("-------------", 3)
    c.display()

    c.left = Given(4)

    print("-------------", 4)
    c.display()

    c.right = Given(5)

    print("-------------", 5)
    c.display()

    print("end of second test")

    # second test

------------- 1
1
end of first test
------------- 3
3
------------- 4
 3
/ 
4 
------------- 5
 3 
/ \
4 5
end of second test


# Stage 2: Basic Insert — No Percolation

In [9]:
class Heap(Given):  # we inherit from the previous stage code

    def __init__(self, key):  # notice the constructor chaining
        super().__init__(key)

    # display is inherited (along with its recursive helper)

    def size(self):  # necessary to determine new insert location

        if self.left == None and self.right == None:
            return 1

        elif self.left == None:
            return 1 + self.right.size()

        elif self.right == None:
            return 1 + self.left.size()

        else:
            return 1 + self.right.size() + self.left.size()

    def insert(self, value):  # this is the insert function

        path = "{0:b}".format(self.size() + 1)

        # it uses the binary representation of heap size to locate
        self.helper(path[1:], value)

    def helper(self, path, value):  # helper for insert

        # if path is one character long: time to add

        if path == "1":  # insert leaf as the right child
            self.right = Heap(value)

        elif path == "0":  # insert leaf as left child
            self.left = Heap(value)

        else:  # otherwise recursively move down the path

            nextStep = path[0]

            if nextStep == '0':
                self.left.helper(path[1:], value)

            else:
                self.right.helper(path[1:], value)


print("-------------", 100)  # insert 100 in empty tree

a = Heap(100)

a.display()

for i in range(99, 92, -1):

    # then keep inserting 99, 98, ... 95, 94 and stop at 93
    print("------------- insert ", i)

    a.insert(i)

    a.display()

# note that at this stage no promotion occurs for smaller numbers

------------- 100
100
------------- insert  99
  100
 /   
99   
------------- insert  98
  100_ 
 /    \
99   98
------------- insert  97
    100_ 
   /    \
  99   98
 /       
97       
------------- insert  96
    __100_ 
   /      \
  99_    98
 /   \     
97  96     
------------- insert  95
    __100___ 
   /        \
  99_      98
 /   \    /  
97  96   95  
------------- insert  94
    __100___   
   /        \  
  99_      98_ 
 /   \    /   \
97  96   95  94
------------- insert  93
      __100___   
     /        \  
    99_      98_ 
   /   \    /   \
  97  96   95  94
 /               
93               


# Stage 3: Insert with Heap Percolation

In [10]:
class Heap(Given):  # exactly as before

    def __init__(self, key):
        super().__init__(key)

    def size(self):  # same as before

        if self.left == None and self.right == None:
            return 1

        elif self.left == None:
            return 1 + self.right.size()

        elif self.right == None:
            return 1 + self.left.size()

        else:
            return 1 + self.right.size() + self.left.size()

    def insert(self, value):  # one line added

        path = "{0:b}".format(self.size() + 1)

        # print(self.size() + 1, "---> ", path)

        self.helper(path[1:], value)

        self.clean()  # this is the new line

    def helper(self, path, value):  # same as before

        if path == "1":
            self.right = Heap(value)

        elif path == "0":
            self.left = Heap(value)

        else:

            nextStep = path[0]

            if nextStep == '0':
                self.left.helper(path[1:], value)

            else:
                self.right.helper(path[1:], value)

    def clean(self):  # this is a new method

        if self.left != None:
            self.left.clean()

        if self.right != None:
            self.right.clean()

        if self.left == None and self.right == None:  # leaf
            pass

        elif self.right == None:  # only one child, on the left

            if self.key > self.left.key:
                (self.left.key, self.key) = (self.key, self.left.key)

        else:  # two children, do you understand why?

            if self.key <= self.left.key and self.key <= self.right.key:
                pass

            elif self.left.key < self.right.key:
                (self.left.key, self.key) = (self.key, self.left.key)

            elif self.left.key > self.right.key:
                (self.right.key, self.key) = (self.key, self.right.key)

            else:
                pass


# start testing to see how smaller numbers percolate up

print("-------------", 100)

a = Heap(100)

a.display()

for i in range(99, 92, -1):

    print("------------- insert ", i)

    a.insert(i)

    a.display()

------------- 100
100
------------- insert  99
  _99
 /   
100  
------------- insert  98
  _98_ 
 /    \
100  99
------------- insert  97
     97_ 
    /   \
  _98  99
 /       
100      
------------- insert  96
     __96_ 
    /     \
  _97_   99
 /    \    
100  98    
------------- insert  95
     __95___ 
    /       \
  _97_     96
 /    \   /  
100  98  99  
------------- insert  94
     __94___   
    /       \  
  _97_     95_ 
 /    \   /   \
100  98  99  96
------------- insert  93
       __93___   
      /       \  
     94_     95_ 
    /   \   /   \
  _97  98  99  96
 /               
100              


# Stage 4: Remove Top — No Downward Percolation Yet

In [11]:
class Heap(Given):

    def __init__(self, key):
        super().__init__(key)

    def size(self):

        if self.left == None and self.right == None:
            return 1

        elif self.left == None:
            return 1 + self.right.size()

        elif self.right == None:
            return 1 + self.left.size()

        else:
            return 1 + self.right.size() + self.left.size()

    def insert(self, value):

        path = "{0:b}".format(self.size() + 1)

        self.helper(path[1:], value)

        self.clean()

    def helper(self, path, value):

        if path == "1":
            self.right = Heap(value)

        elif path == "0":
            self.left = Heap(value)

        else:

            nextStep = path[0]

            if nextStep == '0':
                self.left.helper(path[1:], value)

            else:
                self.right.helper(path[1:], value)

    def clean(self):

        if self.left != None:
            self.left.clean()

        if self.right != None:
            self.right.clean()

        if self.left == None and self.right == None:
            pass

        elif self.right == None:

            if self.key > self.left.key:
                (self.left.key, self.key) = (self.key, self.left.key)

        elif self.left == None:

            if self.key > self.right.key:
                (self.right.key, self.key) = (self.key, self.right.key)

        else:

            if self.key <= self.left.key and self.key <= self.right.key:
                pass

            elif self.left.key < self.right.key:
                (self.left.key, self.key) = (self.key, self.left.key)

            else:
                (self.right.key, self.key) = (self.key, self.right.key)

    # new code added here

    def removeTop(self):

        print("Removing the top.")

        path = "{0:b}".format(self.size())  # find path to the last element

        value = self.helperRemove(path[1:])  # pull the value from that node

        self.key = value  # and place it in the root, then cut value from tree

        p = self

        last = path[-1]

        for c in path[1:-1]:

            if c == "0":
                p = p.left

            else:
                p = p.right

        if last == "0":
            p.left = None

        else:
            p.right = None

        # self.clean()  # we need to percolate the value down if needed

        return self

    # new helper method, helping with remove

    def helperRemove(self, path):

        if path == "1":
            return self.right.key

        elif path == "0":
            return self.left.key

        else:

            nextStep = path[0]

            if nextStep == '0':
                return self.left.helperRemove(path[1:])

            else:
                return self.right.helperRemove(path[1:])


# start testing

print(
    "First some insertions, starting from the empty heap...\n-------------",
    100
)

a = Heap(100)

a.display()

for i in range(99, 92, -1):

    print("------------- insert ", i)

    a.insert(i)

    a.display()


# new test, removing the top five times

print("Now we start removing the top...")

for i in range(5):

    a = a.removeTop()

    a.display()

First some insertions, starting from the empty heap...
------------- 100
100
------------- insert  99
  _99
 /   
100  
------------- insert  98
  _98_ 
 /    \
100  99
------------- insert  97
     97_ 
    /   \
  _98  99
 /       
100      
------------- insert  96
     __96_ 
    /     \
  _97_   99
 /    \    
100  98    
------------- insert  95
     __95___ 
    /       \
  _97_     96
 /    \   /  
100  98  99  
------------- insert  94
     __94___   
    /       \  
  _97_     95_ 
 /    \   /   \
100  98  99  96
------------- insert  93
       __93___   
      /       \  
     94_     95_ 
    /   \   /   \
  _97  98  99  96
 /               
100              
Now we start removing the top...
Removing the top.
    __100___   
   /        \  
  94_      95_ 
 /   \    /   \
97  98   99  96
Removing the top.
    __96___ 
   /       \
  94_     95
 /   \   /  
97  98  99  
Removing the top.
    __99_ 
   /     \
  94_   95
 /   \    
97  98    
Removing the top.
    98_ 
   /  

# Stage 5: Complete RemoveTop with Downward Percolation

In [12]:
class Heap(Given):

    def __init__(self, key):
        super().__init__(key)

    def size(self):

        if self.left == None and self.right == None:
            return 1

        elif self.left == None:
            return 1 + self.right.size()

        elif self.right == None:
            return 1 + self.left.size()

        else:
            return 1 + self.right.size() + self.left.size()

    def insert(self, value):

        path = "{0:b}".format(self.size() + 1)

        self.helper(path[1:], value)

        self.clean()

    def helper(self, path, value):

        if path == "1":
            self.right = Heap(value)

        elif path == "0":
            self.left = Heap(value)

        else:

            nextStep = path[0]

            if nextStep == '0':
                self.left.helper(path[1:], value)

            else:
                self.right.helper(path[1:], value)

    def clean(self):

        if self.left != None:
            self.left.clean()

        if self.right != None:
            self.right.clean()

        if self.left == None and self.right == None:
            pass

        elif self.right == None:

            if self.key > self.left.key:
                (self.left.key, self.key) = (self.key, self.left.key)

        else:

            if self.key <= self.left.key and self.key <= self.right.key:
                pass

            elif self.left.key < self.right.key:
                # what if elif is if here?
                (self.left.key, self.key) = (self.key, self.left.key)

            else:
                (self.right.key, self.key) = (self.key, self.right.key)

    def removeTop(self):

        print("Removing the top:", self.key)

        path = "{0:b}".format(self.size())

        value = self.helperRemove(path[1:])

        self.key = value

        p = self

        last = path[-1]

        for c in path[1:-1]:

            if c == "0":
                p = p.left

            else:
                p = p.right

        if last == "0":
            p.left = None

        else:
            p.right = None

        self.cleanRemove()  # we need to percolate the value down if needed

        return self  # do you know why?

    def helperRemove(self, path):

        if path == "1":
            return self.right.key

        elif path == "0":
            return self.left.key

        else:

            nextStep = path[0]

            if nextStep == '0':
                return self.left.helperRemove(path[1:])

            else:
                return self.right.helperRemove(path[1:])

    def cleanRemove(self):  # this is a different kind of cleaning

        if self.left == None and self.right == None:  # leaf

            pass  # nothing to clean, so stop here

        elif self.right == None:  # only one child, on the left

            if self.key > self.left.key:  # push the value down if necessary

                (self.left.key, self.key) = (self.key, self.left.key)

        else:  # two children (see [1] above)

            if self.key <= self.left.key and self.key <= self.right.key:

                pass  # nothing needs to be pushed down

            if self.left.key < self.right.key:  # swap with the smallest

                (self.left.key, self.key) = (self.key, self.left.key)

                self.left.cleanRemove()  # then keep pushing it down on that side

            else:  # the smallest is on the right side, so swap with that

                (self.right.key, self.key) = (self.key, self.right.key)

                self.right.clean()  # then keep at it like before


print(
    "We start with insertions, from the empty heap...\n-------------",
    100
)

a = Heap(100)

a.display()

for i in range(99, 92, -1):

    print("------------- insert ", i)

    a.insert(i)

    a.display()


print(
    "Now we start removing the top value...\n"
    "... notice how the temporary top percolates down."
)

for i in range(5):

    a = a.removeTop()

    a.display()

We start with insertions, from the empty heap...
------------- 100
100
------------- insert  99
  _99
 /   
100  
------------- insert  98
  _98_ 
 /    \
100  99
------------- insert  97
     97_ 
    /   \
  _98  99
 /       
100      
------------- insert  96
     __96_ 
    /     \
  _97_   99
 /    \    
100  98    
------------- insert  95
     __95___ 
    /       \
  _97_     96
 /    \   /  
100  98  99  
------------- insert  94
     __94___   
    /       \  
  _97_     95_ 
 /    \   /   \
100  98  99  96
------------- insert  93
       __93___   
      /       \  
     94_     95_ 
    /   \   /   \
  _97  98  99  96
 /               
100              
Now we start removing the top value...
... notice how the temporary top percolates down.
Removing the top: 93
     __94___   
    /       \  
  _97_     95_ 
 /    \   /   \
100  98  99  96
Removing the top: 94
     __95___ 
    /       \
  _97_     96
 /    \   /  
100  98  99  
Removing the top: 95
     __96_ 
    /     \


# Stage 6: Handling an Empty Heap

In [13]:
class Heap(Given):

    def __init__(self, key):
        super().__init__(key)

    def size(self):

        if self.left == None and self.right == None:
            return 1

        elif self.left == None:
            return 1 + self.right.size()

        elif self.right == None:
            return 1 + self.left.size()

        else:
            return 1 + self.right.size() + self.left.size()

    def insert(self, value):

        path = "{0:b}".format(self.size() + 1)

        self.helper(path[1:], value)

        self.clean()

    def helper(self, path, value):

        if path == "1":
            self.right = Heap(value)

        elif path == "0":
            self.left = Heap(value)

        else:

            nextStep = path[0]

            if nextStep == '0':
                self.left.helper(path[1:], value)

            else:
                self.right.helper(path[1:], value)

    def clean(self):

        if self.left != None:
            self.left.clean()

        if self.right != None:
            self.right.clean()

        if self.left == None and self.right == None:
            pass

        elif self.right == None:

            if self.key > self.left.key:
                (self.left.key, self.key) = (self.key, self.left.key)

        else:

            if self.key <= self.left.key and self.key <= self.right.key:
                pass

            elif self.left.key < self.right.key:
                (self.left.key, self.key) = (self.key, self.left.key)

            else:
                (self.right.key, self.key) = (self.key, self.right.key)

    def removeTop(self):

        print("Removing the top:", self.key)

        path = "{0:b}".format(self.size())

        value = self.helperRemove(path[1:])

        self.key = value

        p = self

        last = path[-1]

        for c in path[1:-1]:

            if c == "0":
                p = p.left

            else:
                p = p.right

        if last == "0":
            p.left = None

        else:
            p.right = None

        self.cleanRemove()

        return self  # do you know why?

    def helperRemove(self, path):

        if path == "1":
            return self.right.key

        elif path == "0":
            return self.left.key

        else:

            nextStep = path[0]

            if nextStep == '0':
                return self.left.helperRemove(path[1:])

            else:
                return self.right.helperRemove(path[1:])

    def cleanRemove(self):

        if self.left == None and self.right == None:

            pass

        elif self.right == None:

            if self.key > self.left.key:
                (self.left.key, self.key) = (self.key, self.left.key)

        else:

            if self.key <= self.left.key and self.key <= self.right.key:
                pass

            if self.left.key < self.right.key:

                (self.left.key, self.key) = (self.key, self.left.key)

                self.left.cleanRemove()

            else:

                (self.right.key, self.key) = (self.key, self.right.key)

                self.right.clean()


# two wrappers that would not be necessary
# if we used the null object design pattern

def removeTop(heap):

    print("Remove the top value in the heap.")

    if heap == None:

        return None

    elif heap.left == heap.right == None:

        return None

    else:

        return heap.removeTop()


def display(heap):

    print("The heap becomes:")

    if heap == None:

        print(None)

    else:

        heap.display()


# we should probably do something similar with insert


print(
    "We start with insertions, from the empty heap...\n-------------",
    100
)

a = Heap(100)

a.display()

for i in range(99, 92, -1):

    print("------------- insert ", i)

    a.insert(i)

    a.display()


print(
    "Now we start removing the top value...\n"
    "... notice how the temporary top percolates down."
)

size = a.size()

for i in range(size + 3):

    a = removeTop(a)  # see the change here?

    display(a)  # how about here?

    print("----------------------------")

We start with insertions, from the empty heap...
------------- 100
100
------------- insert  99
  _99
 /   
100  
------------- insert  98
  _98_ 
 /    \
100  99
------------- insert  97
     97_ 
    /   \
  _98  99
 /       
100      
------------- insert  96
     __96_ 
    /     \
  _97_   99
 /    \    
100  98    
------------- insert  95
     __95___ 
    /       \
  _97_     96
 /    \   /  
100  98  99  
------------- insert  94
     __94___   
    /       \  
  _97_     95_ 
 /    \   /   \
100  98  99  96
------------- insert  93
       __93___   
      /       \  
     94_     95_ 
    /   \   /   \
  _97  98  99  96
 /               
100              
Now we start removing the top value...
... notice how the temporary top percolates down.
Remove the top value in the heap.
Removing the top: 93
The heap becomes:
     __94___   
    /       \  
  _97_     95_ 
 /    \   /   \
100  98  99  96
----------------------------
Remove the top value in the heap.
Removing the top: 94


# Stage 7: Random Heap Testing

In [14]:
# reminder: these two methods are not part of the class Heap
# instead they're at the top level

def removeTop(heap):

    # print("Remove the top value in the heap.")

    if heap == None:

        return None

    elif heap.left == heap.right == None:

        return None

    else:

        return heap.removeTop()


def display(heap):

    # print("The heap becomes:")

    if heap == None:

        print(None)

    else:

        heap.display()


import random  # final level of testing


b = Heap(random.randrange(-50, 50))  # start with random value

b.display()


for _ in [2, 3, 4, 5, 6, 7, 8, 9]:
    # add eight more to the heap (for a total of nine)

    value = random.randrange(-50, 50)

    print(
        "----------------( now inserting "
        + str(value)
        + " )--"
    )

    b.insert(value)

    b.display()

    # so far so good
    # there was no chance of None thus far


for _ in range(12):
    # then delete 12 times
    # even though we only have 9 in the heap at this time

    print("Removing top of heap now ... heap becomes:")

    b = removeTop(b)

    display(b)

2
----------------( now inserting 48 )--
  2
 / 
48 
----------------( now inserting -47 )--
  -47 
 /   \
48   2
----------------( now inserting -12 )--
    _-47 
   /    \
  -12   2
 /       
48       
----------------( now inserting 23 )--
    ___-47 
   /      \
  -12_    2
 /    \    
48   23    
----------------( now inserting -33 )--
    ___-47__  
   /        \ 
  -12_     -33
 /    \   /   
48   23   2   
----------------( now inserting 26 )--
    ___-47__    
   /        \   
  -12_     -33_ 
 /    \   /    \
48   23   2   26
----------------( now inserting 46 )--
      ___-47__    
     /        \   
    -12_     -33_ 
   /    \   /    \
  46   23   2   26
 /                
48                
----------------( now inserting -31 )--
         ___-47__    
        /        \   
    ___-31_     -33_ 
   /       \   /    \
  -12_    23   2   26
 /    \              
48   46              
Removing top of heap now ... heap becomes:
Removing the top: -47
       ___-33__   
      / 